[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/08_rmsnorm.ipynb)

# 🟡 Medium: Implement RMSNorm

Реализуйте **RMSNorm** (Root Mean Square Normalization). Она стабилизирует масштаб hidden vector по последней dimension, но, в отличие от LayerNorm, не вычитает mean и не добавляет bias.

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot w, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}$$

### Интуиция
RMS измеряет типичную абсолютную величину coordinates без учёта знака. Деление на RMS делает mean square нормализованного vector близким к единице. Weight формы `(D,)` затем обучаемо масштабирует каждую coordinate.

Для input `(B,T,D)` reduction выполняется только по `D`. RMS должен иметь shape `(B,T,1)`, поэтому требуется `keepdim=True`; weight `(D,)` broadcast по batch и sequence.

### Сравнение с LayerNorm
- LayerNorm центрирует через `x-mean`, RMSNorm сохраняет mean и направление смещения;
- обе нормализуют последнюю dimension отдельно для каждого token;
- RMSNorm имеет scale weight, но в этом контракте нет beta shift;
- running statistics и различия train/eval не нужны.

### Контракт
```python
def rms_norm(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    # Normalize over the last dimension. No mean subtraction (unlike LayerNorm).
```

### План реализации
1. Возведите input поэлементно в квадрат.
2. Найдите mean по `dim=-1` с сохранением dimension.
3. Добавьте `eps` и возьмите square root.
4. Разделите input на RMS и умножьте на weight.

### Ограничения
- не используйте готовые norm layers или `F.rms_norm`;
- normalizing axis — `dim=-1`;
- сохраняйте autograd для input и weight.

### Быстрые самопроверки
- output shape равна input shape для 2-D и 3-D tensors;
- при weight=1 результат совпадает с ручным `x / rms`;
- mean square результата примерно единица при достаточно больших значениях относительно eps;
- result совпадает с `F.rms_norm` после самостоятельной реализации;
- gradients существуют для input и weight.

### Типичные ошибки
- вычитание mean, случайно превращающее функцию в LayerNorm;
- mean по всем dimensions;
- потерянный `keepdim`;
- mean абсолютных значений вместо mean squares;
- `eps` добавляется после square root.

### Официальные материалы и примеры
- [`torch.nn.RMSNorm`](https://docs.pytorch.org/docs/stable/generated/torch.nn.RMSNorm.html) — formula, parameters и shapes;
- [`torch.nn.functional.rms_norm`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.rms_norm.html) — functional reference для проверки.

### Вопросы для самопроверки
1. Как RMSNorm изменится, если умножить весь input vector на положительную константу?
2. Почему mean subtraction здесь не нужна по определению?
3. Как `keepdim=True` помогает broadcasting?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def rms_norm(x, weight, eps=1e-6):
    pass  # Replace this

In [ ]:
# 🧪 Debug
x = torch.randn(2, 8)
w = torch.ones(8)
out = rms_norm(x, w)
print("Output shape:", out.shape)
print("RMS of output:", out.pow(2).mean(dim=-1).sqrt())  # should be ~1

In [ ]:
from torch_judge import check
check('rmsnorm')